In [ ]:
import os
import sys
from sys import path
sys.path.append(os.path.expanduser('~/albatros_analysis'))
import numpy as np 
import numba as nb
import time
from scipy import linalg
from scipy import stats
from matplotlib import pyplot as plt
from datetime import datetime as dt
from src.correlations import baseband_data_classes as bdc
from src.utils import baseband_utils as butils
from src.utils import orbcomm_utils as outils
from scipy.optimize import least_squares
import json
#import extra_functions as ef
import random
#import h5py
from scripts.xcorr import helper as hp
import importlib
from scipy.interpolate import interp1d

In [3]:
def pred(coord1, coord2, start_t, end_t, pulse_idx, channel, satID):
    '''
    predicted phase given satellite
    '''
    bench_time = time.time()
    chunk_len = 30000 * (4096/250e6)
    tle_path = outils.get_tle_file(start_t, "/project/rrg-sievers/mohanagr/OCOMM_TLES")
    pulse_len_s = end_t - start_t
    d = outils.get_sat_delay_new(coord1, coord2, tle_path, start_t, pulse_len_s+1, satID)

    pulse_len_chunks = np.ceil(pulse_len_s / chunk_len)
    pulse_freq = outils.chan2freq(channel, alias=True)

    interp_chunk_times = (np.arange(pulse_len_chunks) * chunk_len)

    #get the delay values for each of these chunks
    delay = np.interp(interp_chunk_times, np.arange(len(d)), d)

    #get the predicted phase at each chunk
    pred = (-delay + delay[0]) * 2 * np.pi * pulse_freq
    print("time taken pred", time.time() - bench_time)

    return pred  

In [4]:
config_path = '/home/thomasb/albatros_analysis/scripts/orbcomm'
config_name = 'config.json'

satdet_path = '/scratch/thomasb/pulsedata_1753133403'
satdet_name = "pulsedata_1753133403_1757540615.4208999.json"

T_SPECTRA = 4096/250e6

bline_ants = set({'Antenna 1', 'Antenna 2'})

In [5]:
antennas = []
with open(f"{config_path}/{config_name}", "r") as f:
    config = json.load(f)
    for i, (ant, details) in enumerate(config["antennas"].items()):
        ant_dict = {}
        ant_dict['name'] = details['name']
        ant_dict['coordinates'] = details['coordinates']
        ant_dict['path'] = details['path']

        if i ==0:
            ant_dict['reference'] = 'T'
        else:
            ant_dict['reference'] = 'F'

        if ant_dict['name'] in bline_ants:
            antennas.append(ant_dict)

    global_start_time = config["correlation"]["start_timestamp"]
    global_end_time = config["correlation"]["end_timestamp"]
    v_acclen = config["correlation"]["vis_acclen"]
    
nants = len(antennas)
print(antennas)
print('number of antenna:', nants)
print("Visibility Accumulation Length", v_acclen)
print('global start and end times:', global_start_time, global_end_time)

tle_path = outils.get_tle_file(global_start_time, "/project/rrg-sievers/mohanagr/OCOMM_TLES")
chunk_length = T_SPECTRA * v_acclen

with open(f'{satdet_path}/{satdet_name}') as f:
    data = json.load(f)
    ants = data[f'{global_start_time}']
    for i, ant_from_list in enumerate(antennas):
        if ant_from_list['name'] == 'Antenna 1':
            antennas[i]['consensus_offset'] = 0
            continue
        for j, (ant_from_satdet, details) in enumerate(ants.items()):
            if ant_from_list['name'] == ant_from_satdet:
                antennas[i]['consensus_offset'] = details['consensus_offset']
                antennas[i]['pulse_data'] = details['pulse_data']

for term in antennas:
    print(term)

[{'name': 'Antenna 1', 'coordinates': [79.41718333333333, -90.76735, 189], 'path': '/scratch/mohanagr/summer_2025/baseband/mars1', 'reference': 'T'}, {'name': 'Antenna 2', 'coordinates': [79.41721666666666, -90.75885, 176], 'path': '/scratch/mohanagr/summer_2025/baseband/mars2', 'reference': 'F'}]
number of antenna: 2
Visibility Accumulation Length 30000
global start and end times: 1753133403 1753140603
{'name': 'Antenna 1', 'coordinates': [79.41718333333333, -90.76735, 189], 'path': '/scratch/mohanagr/summer_2025/baseband/mars1', 'reference': 'T', 'consensus_offset': 0}
{'name': 'Antenna 2', 'coordinates': [79.41721666666666, -90.75885, 176], 'path': '/scratch/mohanagr/summer_2025/baseband/mars2', 'reference': 'F', 'consensus_offset': 115507586, 'pulse_data': [{'start': 0, 'end': 305, 'sats_present': {'28654': [[1841, 'UNRELIABLE']], '25338': [[1841, 'UNRELIABLE']]}, 'individual_offset': 115507586, 'diff_to_consensus': 0}, {'start': 0, 'end': 305, 'sats_present': {'28654': [[1841, 'UN

In [8]:
info = antennas[1]
consensus_offset = info['consensus_offset']
print(consensus_offset)
observed_data = {}
desired_pulse_idx = 4

p = info['pulse_data'][desired_pulse_idx]


print(f"---------STARTING PULSE---------")
print(p)
#--------times-----

relative_start_time = p['start']
relative_end_time = p['end']
t_start = global_start_time + relative_start_time
pulse_len_s = relative_end_time - relative_start_time
t_end = t_start + pulse_len_s
pulse_len_chunks = int(np.ceil((pulse_len_s)/(T_SPECTRA * v_acclen)))
print("rel. start, end t:", relative_start_time, relative_end_time)
print("global start t:", t_start)
print("pulse len in s:", pulse_len_s)
print("pulse len in chunks:", pulse_len_chunks)

#-------other data--------
ra_path = antennas[0]['path']
print(ra_path)
nra_path = antennas[1]['path']
print(nra_path)

#----get initialized information----
files_a1, idx1 = butils.get_init_info(t_start, t_end, ra_path)
files_a2, idx2 = butils.get_init_info(t_start, t_end, nra_path)

#-------set up channels-------
channels = bdc.get_header(files_a1[0])["channels"].astype('int64')
chanstart = np.where(channels == 1834)[0][0] 
chanend = np.where(channels == 1852)[0][0]
nchans=chanend-chanstart

#--------call get_avg_fast----------
time_pulse=time.time()
pols, rowcounts, channels = hp.get_avg_fast(ra_path, 
                                            nra_path, 
                                            t_start, 
                                            t_end, 
                                            consensus_offset, 
                                            v_acclen, 
                                            pulse_len_chunks, 
                                            chanstart=chanstart, 
                                            chanend=chanend)

pulse_vis = np.empty((pulse_len_chunks, nchans), dtype=np.complex128)
observed_data[f'{t_start}_{t_end}'] = pulse_vis

print(f"DONE PULSE. TIME:", time.time()-time_pulse)

115507586
---------STARTING PULSE---------
{'start': 3400, 'end': 3740, 'sats_present': {'59051': [[1836, 'RELIABLE'], [1837, 'RELIABLE']]}, 'individual_offset': 115507586, 'diff_to_consensus': 0}
rel. start, end t: 3400 3740
global start t: 1753136803
pulse len in s: 340
pulse len in chunks: 692
/scratch/mohanagr/summer_2025/baseband/mars1
/scratch/mohanagr/summer_2025/baseband/mars2
Not reading any data
took 0.645 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17531/1753136798.raw
took 0.715 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars2/17531/1753136757.raw
before correction 305176 2807617
after correction 305176 2829846
Starting at:  305176 in filenum:  /scratch/mohanagr/summer_2025/baseband/mars1/17531/1753136798.raw for antenna 1
Starting at:  2829846 in filenum:  /scratch/mohanagr/summer_2025/baseband/mars2/17531/1753136757.raw for antenna 2
ACCLEN RECEIVED IS 30000
took 0.715 seconds to read raw data on  /scratch/mohanagr/su

In [ ]:
pulse_info = antennas[1]['pulse_data'][desired_pulse_idx]
start, end = pulse_info['start'] + global_start_time, pulse_info['end'] + global_start_time
sats = list(pulse_info['sats_present'].keys())
print(sats)

vis = observed_data[f'{start}_{end}']
p_vis = np.angle(vis)
amp = np.abs(vis)

chanlist = np.arange(1834, 1852)

mean_amp = []
for i in range(18):
    mean_amp.append(np.mean(amp[:,i]))
chan_small_idx = np.where(mean_amp == np.max(mean_amp))[0][0]

print(chan_small_idx)
chan_big_idx = chanlist[chan_small_idx]
print(chan_big_idx)

phase = np.unwrap(p_vis[:,chan_small_idx]) - p_vis[0, chan_small_idx]
diff = np.diff(phase)

for sat in sats:
    bench = 10e9
    pr = pred(antennas[0]['coordinates'],
                antennas[1]['coordinates'],
                start,
                end,
                desired_pulse_idx, 
                chan_big_idx,
                int(sat))[:len(phase)]
    res = np.linalg.norm(phase - pr)
    if res < bench:
        pred_phase = pr

fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(10, 8))
ax[0,0].set_xlabel("Channel Index (~60kHz interval)")
ax[0,0].set_ylabel("Chunk Number (~0.5s interval)")
ax[0,0].set_title("Wrapped Phase, All Channels")
im = ax[0,0].imshow(p_vis, aspect='auto', cmap='RdBu', interpolation="none")
fig.colorbar(im)

ax[0,1].set_xlabel("Chunk Number (~0.5s interval)")
ax[0,1].set_ylabel("Phase (Radians)")
ax[0,1].set_title("Unwrapped Phase, Detection Channel")
ax[0,1].plot(phase)
ax[0,1].plot(pred_phase)


['59051']
[]
[]
time for one delay pred: 0.031210660934448242


ValueError: operands could not be broadcast together with shapes (692,) (0,) 

[11]
